In [5]:
import sys
!{sys.executable} -m pip install nltk scipy

  Using cached nltk-3.9.4-py3-none-any.whl.metadata (3.2 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached regex-2026.5.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached nltk-3.9.4-py3-none-any.whl (1.6 MB)
Using cached regex-2026.5.9-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (801 kB)
Using cached joblib-1.5.3-py3-none-any.whl (309 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [nltk]━━━━━━ 3/4 [nltk]b]

[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: /home/meron-t-gebru/kaim/climate-challenge-week0/venv/bin/python -m pip install --upgrade pip


In [6]:
# %% [markdown]
# # Task 3: Sentiment-Return Correlation Analysis
# **Author:** Meron Gebru | **Organization:** Nova Financial Solutions  
# *Quantifying the relationship between financial news sentiment and daily stock returns.*

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# Download VADER lexicon (one-time)
import nltk
nltk.download('vader_lexicon', quiet=True)
sia = SentimentIntensityAnalyzer()

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Environment ready")

✅ Environment ready


In [7]:
# %% [markdown]
# ## 1. Load & Align News Data to Trading Days

# %%
# Load raw news data (use only needed columns for memory efficiency)
news = pd.read_csv('../data/raw/raw_analyst_ratings.csv', 
                   usecols=['headline', 'stock', 'date'], 
                   low_memory=False)

# Filter to our 5 target stocks & standardize tickers (FB → META)
TARGETS = ['AAPL', 'GOOG', 'AMZN', 'NVDA', 'META']
news['stock'] = news['stock'].str.upper().replace({'FB': 'META'})
news = news[news['stock'].isin(TARGETS)].copy()

# Parse dates & extract trading day (normalize to date-only)
news['pub_date'] = pd.to_datetime(news['date'], errors='coerce', utc=True)
news['trading_date'] = news['pub_date'].dt.normalize().dt.date

# Handle weekends: shift Sat/Sun articles to next Monday (per assignment)
def shift_weekend_to_trading_day(d):
    if pd.isna(d): 
        return pd.NaT
    day = d.weekday()
    if day == 5:  # Saturday → Monday
        return d + pd.Timedelta(days=2)
    if day == 6:  # Sunday → Monday
        return d + pd.Timedelta(days=1)
    return d

news['trading_date'] = news['trading_date'].apply(shift_weekend_to_trading_day)
news = news.dropna(subset=['trading_date', 'headline'])

print(f"✅ Filtered & aligned: {len(news):,} articles for {len(TARGETS)} tickers")
print(f"📅 Date range: {news['trading_date'].min()} to {news['trading_date'].max()}")

✅ Filtered & aligned: 50 articles for 5 tickers
📅 Date range: 2020-06-01 to 2020-06-10


In [8]:
# %% [markdown]
# ## 2. Sentiment Analysis with VADER

# %%
print("🧠 Scoring headlines with VADER... (~30-60 seconds)")

# VADER compound score: normalized sentiment [-1, 1]
news['vader_compound'] = news['headline'].apply(
    lambda x: sia.polarity_scores(str(x))['compound']
)

# Aggregate: average sentiment per (stock, trading_date)
daily_sentiment = news.groupby(['stock', 'trading_date']).agg(
    avg_sentiment=('vader_compound', 'mean'),
    article_count=('vader_compound', 'count')
).reset_index()

print(f"✅ Aggregated to {len(daily_sentiment):,} daily sentiment records")
print(f"📊 Sample:\n{daily_sentiment.head()}")

🧠 Scoring headlines with VADER... (~30-60 seconds)
✅ Aggregated to 17 daily sentiment records
📊 Sample:
  stock trading_date  avg_sentiment  article_count
0  AAPL   2020-06-09       0.246900              4
1  AAPL   2020-06-10       0.198850              6
2  AMZN   2020-06-09       0.077775              4
3  AMZN   2020-06-10       0.391233              6
4  GOOG   2020-06-04       0.000000              1


In [9]:
# %% [markdown]
# ## 3. Compute Daily Stock Returns

# %%
stock_dfs = []
for ticker in TARGETS:
    df = pd.read_csv(f'../data/raw/{ticker}.csv')
    df['Date'] = pd.to_datetime(df['Date']).dt.date
    
    # Assignment formula: (Close_t - Close_{t-1}) / Close_{t-1} * 100
    df['Daily_Return'] = df['Close'].pct_change() * 100
    df['stock'] = ticker
    stock_dfs.append(df[['Date', 'Daily_Return', 'stock']].dropna())

stock_returns = pd.concat(stock_dfs, ignore_index=True)
print("✅ Daily returns computed for all 5 tickers")
print(f"📈 Sample returns:\n{stock_returns.head()}")

✅ Daily returns computed for all 5 tickers
📈 Sample returns:
         Date  Daily_Return stock
0  2009-01-05      4.220416  AAPL
1  2009-01-06     -1.649399  AAPL
2  2009-01-07     -2.160860  AAPL
3  2009-01-08      1.856959  AAPL
4  2009-01-09     -2.286921  AAPL


In [10]:
# %% [markdown]
# ## 4. Merge & Calculate Pearson Correlation

# %%
# Merge on stock + date
merged = pd.merge(
    daily_sentiment, 
    stock_returns, 
    left_on=['stock', 'trading_date'], 
    right_on=['stock', 'Date'], 
    how='inner'
)
merged = merged.drop(columns=['Date']).dropna()

print(f"🔗 Merged dataset: {len(merged):,} records")

# Calculate Pearson correlation
corr_val, p_val = pearsonr(merged['avg_sentiment'], merged['Daily_Return'])
print(f"\n📈 Pearson Correlation: r = {corr_val:.4f} (p = {p_val:.4f})")

🔗 Merged dataset: 17 records

📈 Pearson Correlation: r = -0.0239 (p = 0.9274)
